# Wimprates Validation

Noble-gas shell spectra compared with the sibling wimprates checkout and saved CSV references.

In [ ]:
import os
import sys
import warnings

sys.path.insert(0, '..')
sys.path.insert(0, '.')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/mpl')

import numpy as np
import matplotlib.pyplot as plt
import numericalunits as nu
from scipy.integrate import simpson

nu.reset_units('SI')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='torchquad')
plt.rcParams['text.usetex'] = False
plt.rcParams['figure.dpi'] = 120

BENCHMARK_MASSES_MEV = [10.0, 100.0, 1000.0]
BENCHMARK_FDM = {'heavy': 0, 'light': 2}
DEFAULT_NE_BINS = [1, 2, 3, 4, 5]

from tests.validation_helpers import load_upstream_wimprates, old_wimprates_csv_path
from DMeRates.DMeRate import DMeRate
from DMeRates.Constants import binding_es

wr = load_upstream_wimprates()
plt.rcParams['text.usetex'] = False
XE_SHELLS = {'5p': (5, 1), '5s': (5, 0), '4d': (4, 2), '4p': (4, 1), '4s': (4, 0)}
AR_SHELLS = ['3s', '3p12']

For noble gases, the minimum velocity includes the shell binding energy: `vmin = (E + Ebind)/q + q/(2mX)`. The threshold diagnostic below reports any nonzero DMeRates support below each shell binding energy.

In [ ]:
obj = DMeRate('Xe', form_factor_type='wimprates', device='cpu')
obj.update_params(238.0, 250.2, 544.0, 0.3e9, 1e-36)
energy = (obj.Earr / nu.eV).detach().cpu().numpy()
drs = obj.noble_dRdE(100.0, 0, 'imb')

fig, axes = plt.subplots(len(XE_SHELLS), 1, figsize=(7, 10), sharex=True)
for ax, (shell, (n, l)) in zip(axes, XE_SHELLS.items()):
    dm = drs[shell].detach().cpu().numpy()
    ref = wr.rate_dme(
        obj.Earr.detach().cpu().numpy(),
        n,
        l,
        mw=100.0 * nu.MeV / nu.c0**2,
        sigma_dme=1e-36 * nu.cm**2,
        f_dm='1',
    )
    ax.plot(energy, dm, label='DMeRates')
    ax.plot(energy, ref, '--', label='wimprates')
    ax.set_yscale('log')
    ax.set_ylabel(shell)
    below = obj.Earr < binding_es['Xe'][shell]
    print(shell, 'max below binding:', float(drs[shell][below].max()) if bool(np.any(below.cpu().numpy())) else 0.0)
axes[-1].set_xlabel('electron recoil energy [eV]')
axes[0].legend()
fig.suptitle('Xe per-shell dR/dE at mX=100 MeV')
plt.show()

In [ ]:
ne_bins = np.arange(1, 21)
rates, shells = obj.calculate_nobleGas_rates([100.0], 'imb', 0, ne_bins, returnShells=True)
rates = rates[0].detach().cpu().numpy()
fig, ax = plt.subplots(figsize=(7, 4))
for i, shell in enumerate(shells):
    ax.plot(ne_bins, rates[:, i], marker='o', label=shell)
ax.set_yscale('log')
ax.set_xlabel('ne')
ax.set_ylabel('events/kg/year')
ax.set_title('Xe dR/dne by shell')
ax.legend(ncol=2, fontsize=8)
plt.show()

In [ ]:
for filename in ['darkside_100MeV_1e-36_fdm0_3p.csv', 'darkside_100MeV_1e-36_fdm0_3s.csv']:
    path = old_wimprates_csv_path(filename)
    if not os.path.exists(path):
        print('missing', path)
        continue
    data = np.loadtxt(path, delimiter=',')
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(data[:, 0], data[:, 1], label=filename)
    ax.set_yscale('log')
    ax.set_xlabel('energy_eV or ne')
    ax.set_ylabel('reference rate')
    ax.legend()
    plt.show()

In [ ]:
obj_ar = DMeRate('Ar', form_factor_type='wimprates', device='cpu')
obj_ar.update_params(238.0, 250.2, 544.0, 0.3e9, 1e-36)
drs_ar = obj_ar.noble_dRdE(100.0, 0, 'imb')
fig, ax = plt.subplots(figsize=(7, 4))
for shell, spectrum in drs_ar.items():
    ax.plot((obj_ar.Earr / nu.eV).detach().cpu().numpy(), spectrum.detach().cpu().numpy(), label=shell)
ax.set_yscale('log')
ax.set_xlabel('electron recoil energy [eV]')
ax.set_ylabel('DMeRates dR/dE')
ax.set_title('Ar DMeRates shell spectra')
ax.legend()
plt.show()
print('The sibling wimprates checkout used here exposes Xe DME shells only, so Ar live parity is deferred to CSV/reference data.')